### OpenAI API key verification (from `.env`)

This notebook checks that `OPENAI_API_KEY` is loaded from a project-local `.env` file (via `python-dotenv`) and prints a masked preview.

### Prereqs

1. Create a file named `.env` in the project root (same folder as this notebook).
2. Add a line like:

```dotenv
OPENAI_API_KEY=...
```

This notebook will **not** print your full key; it only prints a masked preview.

In [20]:
import hashlib
import json
import os
import time
from datetime import datetime, timezone
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI
from PyPDF2 import PdfReader

_ = (hashlib, json, os, time, datetime, timezone, Any, load_dotenv, OpenAI, PdfReader)

In [15]:
# Load variables from a local .env into process environment
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Create a .env file in the project root with OPENAI_API_KEY=..."
    )

# Print a masked preview (never print the full key)
masked = api_key[:7] + "…" + api_key[-4:] if len(api_key) > 12 else "(set, short key)"
print("OPENAI_API_KEY loaded:", masked)

OPENAI_API_KEY loaded: sk-proj…UcsA


In [16]:
# --- Config ---
DOC_PATH = os.path.join("docs", "doc_1.pdf")
EMBEDDINGS_DIR = "embeddings"
OUT_PATH = os.path.join(EMBEDDINGS_DIR, "doc_1.text-embedding-3-small.json")
EMBEDDING_MODEL = "text-embedding-3-small"

# Metadata (from the screenshot)
case_metadata = {
    "title": "Jebuni and Another and Badingu v Mwinibankuro and Another",
    "citation_full": "Jebuni and Another and Badingu v Mwinibankuro and Another (J8/81/2025) [2025] GHASC 46 (23 July 2025)",
    "media_neutral_citation": "[2025] GHASC 46",
    "court": "Supreme Court",
    "case_number": "J8/81/2025",
    "judges": ["Baffoe-Bonnie, JSC", "Amadu, JSC", "Asiedu, JSC", "Kwofie, JSC"],
    "judgment_date": "2025-07-23",
    "judgment_date_display": "23 July 2025",
    "language": "English",
    "summary": "Court granted special leave in a chieftaincy appeal despite procedural delay; dissent held time bar and refusal warranted.",
    "jurisdiction": "Ghana",
}

# --- Load API key and create client ---
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY", "").strip()
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to your .env file.")

client = OpenAI(api_key=api_key)

if not os.path.exists(DOC_PATH):
    raise FileNotFoundError(f"Document not found: {DOC_PATH}")

os.makedirs(EMBEDDINGS_DIR, exist_ok=True)

with open(DOC_PATH, "rb") as f:
    pdf_bytes = f.read()
sha256 = hashlib.sha256(pdf_bytes).hexdigest()

# --- Guard: if embeddings JSON already exists for this exact PDF, skip recompute ---
if os.path.exists(OUT_PATH):
    with open(OUT_PATH, "r", encoding="utf-8") as f:
        existing = json.load(f)
    existing_sha = (existing.get("document") or {}).get("sha256")
    if existing_sha == sha256:
        # Keep metadata in sync for downstream cells
        case_metadata = existing.get("metadata") or case_metadata
        print("Embeddings already exist for this PDF; skipping recompute:", OUT_PATH)
        print("SHA-256:", sha256)
    else:
        print("Embeddings file exists but SHA differs; recomputing:", OUT_PATH)
        print("Existing SHA:", existing_sha)
        print("Current  SHA:", sha256)
        os.remove(OUT_PATH)

if not os.path.exists(OUT_PATH):
    # --- Read PDF + extract per-page text ---
    reader = PdfReader(DOC_PATH)

    pages = []
    for page_idx, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            pages.append({"page": page_idx, "text": text})

    if not pages:
        raise RuntimeError("No text extracted from PDF (it may be scanned images).")

    # --- Chunking (per-page) ---
    def chunk_text(text: str, chunk_chars: int = 2000, overlap: int = 200):
        text = "\n".join([line.rstrip() for line in text.splitlines()]).strip()
        if not text:
            return []
        chunks = []
        start = 0
        while start < len(text):
            end = min(len(text), start + chunk_chars)
            chunk = text[start:end].strip()
            if chunk:
                chunks.append(chunk)
            if end == len(text):
                break
            start = max(0, end - overlap)
        return chunks

    chunks = []
    for p in pages:
        for c in chunk_text(p["text"]):
            chunks.append({"page": p["page"], "text": c})

    if not chunks:
        raise RuntimeError("Chunking produced no chunks.")

    # --- Create embeddings in batches ---
    def batched(seq, batch_size: int):
        for i in range(0, len(seq), batch_size):
            yield seq[i : i + batch_size]

    vectors = []
    BATCH_SIZE = 64
    for batch in batched(chunks, BATCH_SIZE):
        inputs = [b["text"] for b in batch]
        emb = client.embeddings.create(model=EMBEDDING_MODEL, input=inputs)
        for item in emb.data:
            vectors.append(item.embedding)

    if len(vectors) != len(chunks):
        raise RuntimeError(f"Embedding count mismatch: got {len(vectors)} vectors for {len(chunks)} chunks")

    payload = {
        "embedding_model": EMBEDDING_MODEL,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "document": {
            "path": DOC_PATH,
            "filename": os.path.basename(DOC_PATH),
            "sha256": sha256,
            "file_size_bytes": len(pdf_bytes),
            "num_pages": len(reader.pages),
            "extracted_pages": len(pages),
            "chunks": len(chunks),
        },
        "metadata": case_metadata,
        "chunks": [
            {
                "chunk_id": i,
                "page": chunks[i]["page"],
                "text": chunks[i]["text"],
                "embedding": vectors[i],
            }
            for i in range(len(chunks))
        ],
    }

    with open(OUT_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)

    print("Saved embeddings to:", OUT_PATH)
    print("Chunks:", len(chunks), "| Pages with text:", len(pages), "| PDF pages:", len(reader.pages))

Embeddings already exist for this PDF; skipping recompute: embeddings\doc_1.text-embedding-3-small.json
SHA-256: 05edf9869a7a24048cdfcdac6b2c013437c6fab6eb8f9f2a94e42eb3d2cbbac4


In [21]:
# --- Vector store ingestion (all PDFs in docs/) ---
VECTOR_STORE_NAME = "supreme court cases"
DOCS_DIR = "docs"
EMBEDDINGS_DIR = "embeddings"
VECTOR_STORE_DIR = "vector_store"
STORE_SUMMARY_PATH = os.path.join(EMBEDDINGS_DIR, "vector_store.supreme-court-cases.json")

# Per-document metadata (used for local records / reports).
DOCUMENT_METADATA_BY_FILENAME = {
    "doc_2.pdf": {
        "title": "Akufo-addo and Others Vrs Mahama and Others",
        "citation_full": "Akufo-addo and Others Vrs Mahama and Others [2013] GHASC 3 (29 August 2013)",
        "media_neutral_citation": "[2013] GHASC 3",
        "court": "Supreme Court",
        "judges": [
            "Akuffo, JSC",
            "Anin-Yeboah, JSC",
            "Baffoe-Bonnie, JSC",
            "Ansah, JSC",
            "Adinyira, JSC",
        ],
        "judgment_date": "2013-08-29",
        "judgment_date_display": "29 August 2013",
        "language": "English",
        "jurisdiction": "Ghana",
    }
}

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY", "").strip()
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to your .env file.")

# Reuse the existing client if earlier cells ran; otherwise create one here.
if "client" in globals():
    client = globals()["client"]
else:
    client = OpenAI(api_key=api_key)

os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
os.makedirs(VECTOR_STORE_DIR, exist_ok=True)

# Keep vector store metadata simple and API-safe (strings only).
VECTOR_STORE_METADATA = {
    "jurisdiction": "Ghana",
    "court": "Supreme Court",
}

def get_or_create_vector_store_id(name: str, metadata: dict) -> str:
    existing_id = None
    try:
        vs_list = client.vector_stores.list(limit=100)
        for vs in getattr(vs_list, "data", []) or []:
            if (getattr(vs, "name", "") or "") == name:
                existing_id = vs.id
                break
    except Exception:
        existing_id = None

    if existing_id:
        print("Reusing vector store:", name, "| id:", existing_id)
        return existing_id

    vector_store = client.vector_stores.create(
        name=name, metadata={str(k): str(v) for k, v in (metadata or {}).items()}
    )
    print("Created vector store:", name, "| id:", vector_store.id)
    return vector_store.id

def iter_vector_store_files(vector_store_id: str, limit: int = 100):
    after = None
    while True:
        kwargs = {"vector_store_id": vector_store_id, "limit": limit}
        if after:
            kwargs["after"] = after
        page = client.vector_stores.files.list(**kwargs)
        items = getattr(page, "data", []) or []
        if not items:
            return
        for item in items:
            yield item
        if not bool(getattr(page, "has_more", False)):
            return
        after = getattr(items[-1], "id", None)
        if not after:
            return

def list_existing_filenames(vector_store_id: str) -> dict:
    """Return dict: lower(filename) -> uploaded_file_id."""
    out = {}
    for vs_file in iter_vector_store_files(vector_store_id):
        # Note: in current SDK responses, `id` is the uploaded file id (e.g., file-...).
        file_id = getattr(vs_file, "file_id", None) or getattr(vs_file, "id", None)
        if not file_id:
            continue
        try:
            fobj = client.files.retrieve(file_id)
        except Exception:
            continue
        fn = (getattr(fobj, "filename", "") or "").strip()
        if fn:
            out[fn.lower()] = file_id
    return out

vector_store_id = get_or_create_vector_store_id(VECTOR_STORE_NAME, VECTOR_STORE_METADATA)

# Persist a lightweight store summary used by the app (id + name).
store_summary = {
    "vector_store_name": VECTOR_STORE_NAME,
    "vector_store_id": vector_store_id,
    "updated_at": datetime.now(timezone.utc).isoformat(),
    "note": "Auto-generated by notebook ingestion cell.",
}
with open(STORE_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(store_summary, f, ensure_ascii=False, indent=2)
print("Saved vector store summary to:", STORE_SUMMARY_PATH)

existing_by_name = list_existing_filenames(vector_store_id)
print("Vector store currently contains", len(existing_by_name), "file(s).")

docs_path = os.path.join(DOCS_DIR)
if not os.path.isdir(docs_path):
    raise FileNotFoundError(f"Missing docs folder: {docs_path}")

pdf_paths = []
for name in os.listdir(docs_path):
    if name.lower().endswith(".pdf"):
        pdf_paths.append(os.path.join(docs_path, name))
pdf_paths.sort(key=lambda p: os.path.basename(p).lower())

if not pdf_paths:
    raise RuntimeError(f"No PDFs found in {docs_path!r}.")

added = 0
skipped = 0

for pdf_path in pdf_paths:
    filename = os.path.basename(pdf_path)
    stem, _ = os.path.splitext(filename)
    guard_path = os.path.join(VECTOR_STORE_DIR, f"{stem}.vector_store_record.json")

    doc_metadata = DOCUMENT_METADATA_BY_FILENAME.get(filename) or {}
    doc_metadata_api = {str(k): str(v) for k, v in (doc_metadata or {}).items() if not isinstance(v, (list, dict))}

    with open(pdf_path, "rb") as f:
        pdf_bytes = f.read()
    doc_sha256 = hashlib.sha256(pdf_bytes).hexdigest()

    # Local guard: if we already wrote a record for this filename, skip.
    if os.path.exists(guard_path):
        print("Local guard found; skipping:", filename)
        skipped += 1
        continue

    # Remote safe-check: if the same filename already exists in the vector store, don't upload again.
    existing_file_id = existing_by_name.get(filename.lower())
    if existing_file_id:
        record = {
            "vector_store_name": VECTOR_STORE_NAME,
            "vector_store_id": vector_store_id,
            "file_id": existing_file_id,
            "file_batch_id": None,
            "created_at": datetime.now(timezone.utc).isoformat(),
            "document_path": pdf_path,
            "document_filename": filename,
            "document_sha256": doc_sha256,
            "metadata": doc_metadata,
            "metadata_api": doc_metadata_api,
            "note": f"Skipped upload: '{filename}' already exists in vector store.",
        }
        with open(guard_path, "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)
        print("Remote duplicate detected; wrote local guard:", guard_path)
        skipped += 1
        continue

    # Upload + attach to vector store
    with open(pdf_path, "rb") as f:
        uploaded = client.files.create(file=f, purpose="assistants")
    file_id = uploaded.id
    print("Uploaded:", filename, "| file id:", file_id)

    file_batch = client.vector_stores.file_batches.create(vector_store_id=vector_store_id, file_ids=[file_id])
    file_batch_id = file_batch.id
    print("Started file batch:", file_batch_id)

    status = getattr(file_batch, "status", None)
    t0 = time.time()
    while status not in {"completed", "failed", "cancelled"}:
        time.sleep(1.0)
        file_batch = client.vector_stores.file_batches.retrieve(
            vector_store_id=vector_store_id, batch_id=file_batch_id
        )
        status = getattr(file_batch, "status", None)
        if time.time() - t0 > 300:
            raise TimeoutError("Timed out waiting for vector store file processing (5 minutes).")

    print("Batch status:", status)
    if status != "completed":
        raise RuntimeError(f"Vector store ingestion did not complete successfully: {status}")

    record = {
        "vector_store_name": VECTOR_STORE_NAME,
        "vector_store_id": vector_store_id,
        "file_id": file_id,
        "file_batch_id": file_batch_id,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "document_path": pdf_path,
        "document_filename": filename,
        "document_sha256": doc_sha256,
        "metadata": doc_metadata,
        "metadata_api": doc_metadata_api,
        "note": "Uploaded and ingested into vector store.",
    }
    with open(guard_path, "w", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False, indent=2)
    print("Saved local guard:", guard_path)

    existing_by_name[filename.lower()] = file_id
    added += 1

print("Done.")
print("Added:", added, "| Skipped:", skipped)

Reusing vector store: supreme court cases | id: vs_69a1c8a5c6608191a6bf0d0d815aab5f
Saved vector store summary to: embeddings\vector_store.supreme-court-cases.json
Vector store currently contains 1 file(s).
Local guard found; skipping: doc_1.pdf
Uploaded: doc_2.pdf | file id: file-J46SU525b16JLWyWTYcXgc
Started file batch: vsfb_ibj_69a1d82910d481f489b12e1e2b7df871
Batch status: completed
Saved local guard: vector_store\doc_2.vector_store_record.json
Done.
Added: 1 | Skipped: 1


In [18]:
# --- Write a human-readable report (doc_1) ---
DOC1_EMB_PATH = os.path.join("embeddings", "doc_1.text-embedding-3-small.json")
DOC1_VS_PATH = os.path.join("vector_store", "doc_1.vector_store_record.json")
OUT_TXT = os.path.join("vector_store", "doc_1_embbeding details.txt")

if not os.path.exists(DOC1_EMB_PATH):
    raise FileNotFoundError(f"Missing embeddings file: {DOC1_EMB_PATH}")
if not os.path.exists(DOC1_VS_PATH):
    raise FileNotFoundError(f"Missing vector store record: {DOC1_VS_PATH}")

with open(DOC1_EMB_PATH, "r", encoding="utf-8") as f:
    emb = json.load(f)
with open(DOC1_VS_PATH, "r", encoding="utf-8") as f:
    vs = json.load(f)

doc = emb.get("document", {})
meta = emb.get("metadata", {})
chunks = emb.get("chunks", [])
chunk_count = len(chunks) if isinstance(chunks, list) else None
embedding_dims = None
if isinstance(chunks, list) and chunks and isinstance(chunks[0].get("embedding"), list):
    embedding_dims = len(chunks[0]["embedding"])

doc_sha = (doc.get("sha256") or "").strip()

def report_already_matches(path: str, sha256: str) -> bool:
    if not sha256:
        return False
    if not os.path.exists(path):
        return False
    try:
        with open(path, "r", encoding="utf-8") as f:
            prefix = f.read(50_000)
    except Exception:
        return False
    return f"SHA-256: {sha256}" in prefix

# --- Safe guard: if report already exists for this exact document SHA, skip re-creating it ---
if report_already_matches(OUT_TXT, doc_sha):
    print("Report already exists for this document; skipping:", OUT_TXT)
else:
    judges = meta.get("judges")
    judges_text = ", ".join(judges) if isinstance(judges, list) else (judges or "")

    lines = []
    lines.append("DOC_1 — Embeddings + Vector Store Details")
    lines.append("================================================")
    lines.append("")

    lines.append("1) Document")
    lines.append("-----------")
    lines.append(f"Source file: {doc.get('path', 'docs/doc_1.pdf')}")
    lines.append(f"Filename: {doc.get('filename', 'doc_1.pdf')}")
    if doc.get("sha256"):
        lines.append(f"SHA-256: {doc.get('sha256')}")
    if doc.get("file_size_bytes") is not None:
        lines.append(f"File size (bytes): {doc.get('file_size_bytes')}")
    if doc.get("num_pages") is not None:
        lines.append(f"PDF pages: {doc.get('num_pages')}")
    if doc.get("extracted_pages") is not None:
        lines.append(f"Pages with extracted text: {doc.get('extracted_pages')}")
    if doc.get("chunks") is not None:
        lines.append(f"Chunks created: {doc.get('chunks')}")
    lines.append("")

    lines.append("2) Case metadata (Supreme Court)")
    lines.append("------------------------------")
    lines.append(f"Title: {meta.get('title', '')}")
    lines.append(f"Court: {meta.get('court', '')}")
    lines.append(f"Jurisdiction: {meta.get('jurisdiction', '')}")
    lines.append(f"Case number: {meta.get('case_number', '')}")
    lines.append(f"Media neutral citation: {meta.get('media_neutral_citation', '')}")
    lines.append(f"Full citation: {meta.get('citation_full', '')}")
    lines.append(f"Judgment date: {meta.get('judgment_date_display', meta.get('judgment_date', ''))}")
    lines.append(f"Language: {meta.get('language', '')}")
    if judges_text:
        lines.append(f"Judges: {judges_text}")
    if meta.get("summary"):
        lines.append("Summary:")
        lines.append(str(meta.get("summary")))
    lines.append("")

    lines.append("3) Embeddings artifact")
    lines.append("---------------------")
    lines.append(f"Embedding model: {emb.get('embedding_model', '')}")
    lines.append(f"Created at (UTC): {emb.get('created_at', '')}")
    lines.append(f"Saved JSON: {DOC1_EMB_PATH}")
    if chunk_count is not None:
        lines.append(f"Embedded chunks: {chunk_count}")
    if embedding_dims is not None:
        lines.append(f"Embedding dimensions: {embedding_dims}")
    lines.append("Note: This report does NOT include raw vectors; they are stored in the JSON file.")
    lines.append("")

    lines.append("4) Vector store")
    lines.append("---------------")
    lines.append(f"Vector store name: {vs.get('vector_store_name', '')}")
    lines.append(f"Vector store id: {vs.get('vector_store_id', '')}")
    lines.append(f"Uploaded file id: {vs.get('file_id', '')}")
    lines.append(f"File batch id: {vs.get('file_batch_id', '')}")
    lines.append(f"Created at (UTC): {vs.get('created_at', '')}")
    lines.append(f"Vector store record JSON: {DOC1_VS_PATH}")
    lines.append("")

    lines.append("5) Notes")
    lines.append("--------")
    lines.append("- API keys are not stored in any of these artifacts.")
    lines.append("- Vector store metadata is stored as strings (per API requirements); the record JSON keeps both structured and API-safe forms.")
    lines.append("")

    os.makedirs("vector_store", exist_ok=True)
    with open(OUT_TXT, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("Saved report to:", OUT_TXT)

Report already exists for this document; skipping: vector_store\doc_1_embbeding details.txt
